# Investigation: Dimensional fact selection — priority rule

## Background

For a given concept + period, an XBRL filing may contain multiple facts:

| dimensions | meaning |
|---|---|
| `{}` (none) | **consolidated group-level figure** — the primary financial statement value |
| `{ConsolidatedEntitiesAxis: ParentCompanyMember}` | parent-company standalone (supplemental schedule) |
| `{StatementBusinessSegmentsAxis: SegmentXMember}` | segment breakdown |
| any other non-empty dimensions | filtered sub-total — supplemental disclosure |

**Priority rule:** always prefer the fact with **no dimensions**. A dimensioned fact is a scoped sub-total, not the primary figure. Fall back to a dimensioned fact only if no plain fact exists for that concept+period.

## Concrete example

JPMorgan 10-K 2024 reports `us-gaap:Assets` twice for 2024-12-31:
- `4,002,814` with `{}` → **consolidated total assets** ✅ correct primary value  
- `669,614` with `{ConsolidatedEntitiesAxis: ParentCompanyMember}` → parent-company-only standalone balance sheet (supplemental)

The correct total assets is **4,002,814**.

This notebook verifies that the `statement_viewer`'s `_values_at` implements this rule correctly, and documents the remaining problem: note tables that are **entirely scoped** to a sub-entity (e.g. `Balance Sheets (Details)` = the parent-only schedule) show structurally inconsistent data because `_values_at` always returns the consolidated value even when the table's pres arcs declare a hypercube scope.

In [ ]:
import sys
from collections import defaultdict
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('../src').resolve()))
from xbrl_extraction import Document

doc    = Document.load(Path('../data/output/rawdata_us_16166_16166_XBRL_2025-03-12.json'))
PERIOD = '2024-12-31'

print(f'Filing : {doc.filing.form} FY{doc.filing.fiscal_year}  ({doc.filing.source_file})')
print(f'Facts  : {len(doc.facts)}')

matching_ctxs = {
    cid for cid, ctx in doc.periods.items()
    if (ctx.type == 'instant' and ctx.date == PERIOD)
    or (ctx.type == 'duration' and ctx.end  == PERIOD)
}
print(f'Contexts for {PERIOD}: {len(matching_ctxs)}')

## 1 — All fact instances for `us-gaap:Assets`

Demonstrates why the priority rule matters: many values exist, only one is the primary figure.

In [ ]:
pd.set_option('display.max_colwidth', 120)

rows = [
    {
        'value'     : f.value,
        'dimensions': str(f.dimensions) if f.dimensions else '(none)  ← PRIMARY',
    }
    for f in doc.facts
    if f.concept == 'us-gaap:Assets' and f.period in matching_ctxs
]
df = pd.DataFrame(rows).sort_values('value', ascending=False)
print(f'us-gaap:Assets — {len(df)} instances for {PERIOD}:')
df

## 2 — Verify `_values_at` implements the correct priority rule

In [ ]:
def _values_at(doc, period_end):
    """
    Returns concept -> authoritative value for period_end.

    Priority rule: plain (no-dimension) facts represent the consolidated
    group-level figure and always take precedence over dimensioned facts.
    Dimensioned facts are only used as a fallback when no plain fact exists.
    """
    matching = {
        cid for cid, ctx in doc.periods.items()
        if (ctx.type == 'instant' and ctx.date == period_end)
        or (ctx.type == 'duration' and ctx.end  == period_end)
    }
    plain    = {}  # no dimensions  → consolidated primary figure
    fallback = {}  # has dimensions → scoped sub-total, supplemental only
    for f in doc.facts:
        if f.period not in matching:
            continue
        if not f.dimensions:
            plain.setdefault(f.concept, f.value)
        else:
            fallback.setdefault(f.concept, f.value)
    # plain wins; fallback fills gaps for concepts with no undimensioned fact
    return {**fallback, **plain}

vals = _values_at(doc, PERIOD)

# Spot-check key concepts
spot = ['us-gaap:Assets', 'us-gaap:Liabilities', 'us-gaap:StockholdersEquity',
        'us-gaap:LiabilitiesAndStockholdersEquity', 'us-gaap:CashAndDueFromBanks']

rows = []
for c in spot:
    plain_v = next((f.value for f in doc.facts
                    if f.concept == c and f.period in matching_ctxs and not f.dimensions), None)
    chosen  = vals.get(c)
    rows.append({
        'concept'          : c.split(':')[1],
        'plain (no-dim)'   : plain_v,
        '_values_at result': chosen,
        'correct?'         : 'YES' if chosen == plain_v else 'NO — using fallback (no plain fact exists)',
    })

pd.DataFrame(rows)

## 3 — The remaining problem: note tables scoped to a sub-entity

The priority rule is correct for **primary financial statements** (consolidated balance sheet, income statement, etc.).

However, some note tables are **entirely scoped** to a sub-entity via an XBRL hypercube. Example: `Balance Sheets (Details)` in this filing is the **parent-company-only standalone balance sheet** — every fact in the table should carry `ConsolidatedEntitiesAxis = ParentCompanyMember`.

When `_values_at` is applied to such a table, it correctly returns the consolidated total for `us-gaap:Assets` (4,002,814) — but the component line items in that note (`jpm:TradingAssets`, `jpm:InvestmentsInSubsidiaries`, etc.) have **no plain fact** and fall back to their `ParentCompanyMember`-scoped values. The result is a structurally inconsistent table: components sum to ~668k but the total shows 4,002k.

This is a **display-layer problem**, not a data problem. The correct consolidated Assets (4,002,814) is right; the parent-only note table is simply the wrong context to apply `_values_at` without a scope filter.

In [ ]:
ROLE = 'ParentCompanyBalanceSheetsDetails'

# Components in this note table and their values under _values_at
note_concepts = [
    ('us-gaap:CashAndDueFromBanks'       , 'Cash and due from banks'),
    ('jpm:DepositsWithBankingSubsidiaries', 'Deposits with banking subsidiaries'),
    ('jpm:TradingAssets'                 , 'Trading assets'),
    ('jpm:AdvancesToSubsidiaries'        , 'Advances to subsidiaries'),
    ('jpm:InvestmentsInSubsidiaries'     , 'Investments in subsidiaries'),
    ('us-gaap:OtherAssets'               , 'Other assets'),
    ('us-gaap:Assets'                    , 'Total assets'),
]

def plain_value(concept):
    for f in doc.facts:
        if f.concept == concept and f.period in matching_ctxs and not f.dimensions:
            return f.value
    return None

def dim_value(concept, axis, member):
    for f in doc.facts:
        if (f.concept == concept and f.period in matching_ctxs
                and f.dimensions.get(axis) == member and len(f.dimensions) == 1):
            return f.value
    return None

AXIS   = 'srt:ConsolidatedEntitiesAxis'
MEMBER = 'srt:ParentCompanyMember'

rows = []
for concept, label in note_concepts:
    pv = plain_value(concept)
    dv = dim_value(concept, AXIS, MEMBER)
    chosen = vals.get(concept)
    rows.append({
        'label'                    : label,
        'plain (consolidated)'     : pv,
        'ParentCompanyMember scope': dv,
        '_values_at picks'         : chosen,
        'note: plain preferred?'   : 'yes' if chosen == pv and pv is not None else 'no plain — fallback used',
    })

df_note = pd.DataFrame(rows)
display(df_note)

comp_sum = sum(vals.get(c, 0) for c, _ in note_concepts[:-1])
total    = vals.get('us-gaap:Assets')
print(f'\nSum of component lines (_values_at): {comp_sum:>12,.0f}')
print(f'Total assets         (_values_at): {total:>12,.0f}')
print(f'Apparent gap                     : {total - comp_sum:>12,.0f}')
print()
print('The gap is expected: components use fallback (parent-only scope),')
print('while Assets correctly uses plain (consolidated total).')
print('This note table cannot be rendered consistently using only _values_at.')

## 4 — Identifying hypercube-scoped roles

A note table is scoped to a sub-entity when its pres tree declares an `Axis` (hanging off the `Table`) that has exactly one leaf `Member`. That single member is the required dimension context for every fact in the table.

In [ ]:
def infer_primary_dims(doc, role_short):
    """
    Return {axis: member} for axes that scope the entire table to one
    sub-entity/segment (exactly one leaf member under the axis).
    Axes with 2+ leaf members are within-row breakdowns — excluded.
    """
    arcs = [a for a in doc.pres.arcs if a.role_short == role_short]
    cbp  = defaultdict(list)
    for a in arcs:
        cbp[a.parent].append(a.child)
    all_nodes = {a.parent for a in arcs} | {a.child for a in arcs}
    tables    = [n for n in all_nodes if n.split(':')[-1].endswith('Table')]
    primary   = {}
    for table in tables:
        for axis in cbp.get(table, []):
            if not axis.split(':')[-1].endswith('Axis'):
                continue
            members = []
            def collect(node, members=members):
                for child in cbp.get(node, []):
                    if child.split(':')[-1].endswith('Member'):
                        members.append(child)
                    collect(child)
            collect(axis)
            leaf = [m for m in members if not cbp.get(m)]
            if len(leaf) == 1:
                primary[axis] = leaf[0]
    return primary

# Show all roles in this filing that have a primary scoping dimension
roles = {a.role_short for a in doc.pres.arcs}
rows  = []
for rs in sorted(roles):
    rd   = next((a.role_definition for a in doc.pres.arcs if a.role_short == rs), rs)
    dims = infer_primary_dims(doc, rs)
    if dims:
        rows.append({
            'definition'  : rd[:65],
            'primary dims': ', '.join(
                f"{ax.split(':')[-1]}={mem.split(':')[-1]}"
                for ax, mem in dims.items()
            ),
        })

df_scoped = pd.DataFrame(rows)
print(f'{len(df_scoped)} role(s) with a primary scoping dimension:')
df_scoped

## Summary

### The correct priority rule

When multiple facts share the same concept + period:

1. **No dimensions** → use this. It is the consolidated group-level primary figure.
2. **Any dimensions** → supplemental disclosure only. Use as fallback if no plain fact exists.

This is what `_values_at` in `statement_viewer` already implements (`{**fallback, **plain}`).

### The residual display problem

Some note tables are **entirely scoped** to a sub-entity via an XBRL hypercube (e.g. `Balance Sheets (Details)` = parent-company-only schedule). For these tables:

- `us-gaap:Assets` has a plain fact (4,002,814 consolidated) → `_values_at` correctly picks it.
- Component line items (e.g. `jpm:TradingAssets`) have **no plain fact** → `_values_at` falls back to the parent-scoped value (43,214).
- Result: the table mixes scopes — consolidated total vs parent-only components.

This is not a bug in the priority rule. It is a fundamental mismatch between what `_values_at` provides (consolidated primary figures) and what the note table is designed to display (parent-only figures). Fixing it requires identifying hypercube-scoped roles via `infer_primary_dims` and applying a scope-aware lookup — but only for those roles, and only as a secondary rendering path that does **not** override the plain-fact priority rule for primary statements.